In [15]:
import pandas as pd
import xarray as xr
import numpy as np

from internalizer import Internalizer
from internalizer.calculation_setup import default_setup
from internalizer.regionalization import combine_shares_and_costs, REMIND_REGIONS

In [2]:
EI_VERSION = "3.10"

mifpath = "/p/tmp/davidba/internalization_develop/remind/output/SSP2-NPi-internalize-test-iterative_2025-07-31_14.13.45/lca/remind_runs/remind_SSP2-NPi-internalize-test-iterative-postsolve.mif"
gdxpath = "/p/tmp/davidba/internalization_develop/remind/output/SSP2-NPi-internalize-test-iterative_2025-07-31_14.13.45/fulldata_postsolve.gdx"
pathway = "SSP2-NPi-internalize-test-iterative-postsolve"
monetization = 0.5


In [3]:
IMPACT_CATEGORIES_MC = [
    "acidification",
    "climate change",
    "ecotoxicity",
    "eutrophication",
    "fossil resources",
    "human toxicity",
    "ionizing radiation",
    "land use",
    "metal/mineral resources",
    "ozone depletion",
    "particulate matter formation",
    "photochemical oxidant formation",
    "water use"
]

In [4]:
# set up brightway project
bw_project = f"internalizer_ei_{EI_VERSION}"

In [5]:
# initialize Internalizer
I = Internalizer(
    mifpath,
    "remind",
    pathway,
    EI_VERSION,
    bw_project,
    gdxpath,
    outputfolder = "lca"
)
print(I.scenario)

SSP2-NPi-internalize-test-iterative-postsolve


In [6]:
# premise runs
years = [2020, 2050]
I.years = years
# I.run_premise(years, multiprocessing=True)

In [ ]:
def fill_shares_from_mif(mapping, mifpath, year):
    mif = pd.read_csv(mifpath, sep=";")

    
    

In [38]:
mapping = pd.read_csv("../internalizer/data/mappings/demFE.csv", sep=";")
mapping

,REMIND index,scenario variable,dataset name,dataset reference product,dataset unit,share
0,build - fehes,FE|Buildings|Heating|District Heating,"market for heat, district or industrial, natur...","heat, district or industrial, natural gas",megajoule,regional
1,build - fehes,FE|Buildings|Heating|Electricity|Heat pump,"heat production, air-water heat pump 10kW","heat, air-water heat pump 10kW",megajoule,regional
2,build - feels,FE|Buildings|Heating|Electricity|Resistance,"heat, residential, electric storage heater, us...","heat, from residential heating system",megajoule,regional
3,build - fegas,FE|Buildings|Heating|Gases,"heat production, natural gas, at boiler conden...","heat, central or small-scale, natural gas",megajoule,regional
4,build - feh2s,FE|Buildings|Heating|Hydrogen,"heat, residential, by combustion of hydrogen u...","heat, from residential heating system",megajoule,regional
...,...,...,...,...,...,...
212,trans - fepet,FE|Transport|Pass|Road|LDV|Two Wheelers|Motorc...,"petrol, synthetic, burned in motorcycle",heat,megajoule,regional
213,trans - feelt,FE|Transport|Pass|Road|LDV|Two Wheelers|Motorc...,"electricity, used in battery electric motorcycle","electricity, low voltage",megajoule,regional
214,trans - fepet,FE|Transport|Pass|Road|LDV|Two Wheelers|Motorc...,"bioethanol, burned in motorcycle",heat,megajoule,regional
215,trans - fepet,FE|Transport|Pass|Road|LDV|Two Wheelers|Motorc...,"petrol, burned in motorcycle",heat,megajoule,regional


In [39]:
dflist = []
for region in REMIND_REGIONS:
    df = mapping.copy()
    df["region"] = region
    dflist.append(df)

regionalized_mapping = pd.concat(dflist).set_index(["scenario variable", "region"])

In [40]:
len(mapping["scenario variable"].unique())

217

In [41]:
mif = pd.read_csv(mifpath, sep=";").rename(columns={"Region": "region", "Variable": "scenario variable"})

In [42]:
mifdata = mif.set_index(["scenario variable", "region"])["2050"]

In [43]:
combined_idx = regionalized_mapping.index.intersection(mifdata.index)

In [46]:
sel = regionalized_mapping.loc[combined_idx]

In [47]:
sel["weight"] = mifdata.loc[combined_idx]

In [48]:
sel = sel.reset_index()

In [ ]:
sel["total"] = sel.reset_index().groupby(["REMIND index", "region"])["weight"].transform("sum")


In [50]:
sel["share"] = sel["weight"] / sel["total"]

In [53]:
sel

,scenario variable,region,REMIND index,dataset name,dataset reference product,dataset unit,share,weight,total
0,FE|Buildings|Heating|District Heating,CAZ,build - fehes,"market for heat, district or industrial, natur...","heat, district or industrial, natural gas",megajoule,0.429628,0.059282,0.137984
1,FE|Buildings|Heating|Electricity|Heat pump,CAZ,build - fehes,"heat production, air-water heat pump 10kW","heat, air-water heat pump 10kW",megajoule,0.570372,0.078702,0.137984
2,FE|Buildings|Heating|Electricity|Resistance,CAZ,build - feels,"heat, residential, electric storage heater, us...","heat, from residential heating system",megajoule,0.012193,0.008722,0.715307
3,FE|Buildings|Heating|Gases,CAZ,build - fegas,"heat production, natural gas, at boiler conden...","heat, central or small-scale, natural gas",megajoule,1.000000,1.809625,1.809625
4,FE|Buildings|Heating|Hydrogen,CAZ,build - feh2s,"heat, residential, by combustion of hydrogen u...","heat, from residential heating system",megajoule,NaN,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...
2419,FE|Transport|Pass|Road|LDV|Four Wheelers|Mediu...,USA,trans - fedie,"diesel, synthetic, burned in passenger car",heat,megajoule,0.000736,0.003747,5.091636
2420,FE|Transport|Pass|Road|LDV|Two Wheelers|Motorc...,USA,trans - feelt,"electricity, used in battery electric motorcycle","electricity, low voltage",megajoule,0.009472,0.014673,1.549210
2421,FE|Transport|Pass|Road|LDV|Two Wheelers|Motorc...,USA,trans - fepet,"bioethanol, burned in motorcycle",heat,megajoule,0.000062,0.001010,16.284122
2422,FE|Transport|Pass|Road|LDV|Two Wheelers|Motorc...,USA,trans - fepet,"petrol, burned in motorcycle",heat,megajoule,0.001265,0.020594,16.284122


In [52]:
sel.groupby(["REMIND index", "region"])["share"].sum()

REMIND index   region
build - feels  CAZ       1.0
               CHA       1.0
               EUR       1.0
               IND       1.0
               JPN       1.0
                        ... 
trans - fepet  NEU       1.0
               OAS       1.0
               REF       1.0
               SSA       1.0
               USA       1.0
Name: share, Length: 288, dtype: float64

In [11]:
#
# mapping = ["./mappings/test_PV.csv"]
I.calculate_costs(monetization)

/p/tmp/davidba/internalizer/internalizer/regionalization.py:263: PerformanceWarning: indexing past lexsort depth may impact performance.
  sel = df.loc[i]
/p/tmp/davidba/internalizer/internalizer/regionalization.py:235: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rdf["region"] = region
/p/tmp/davidba/internalizer/internalizer/regionalization.py:235: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rdf["region"] = region
/p/tmp/davidba/internalizer/internalizer/regionalization.py:235: SettingWithCopyWarning: 

In [8]:
impacts = pd.read_csv("/p/tmp/davidba/internalizer/dev/lca/remind/SSP2-NPi-internalize-test-iterative-postsolve/2050/impacts.csv")
impacts = impacts[impacts["dataset location"] == "AT"]
impacts = impacts[impacts["LCIA method"].str.contains("ReCiPe 2016")]
impacts.sort_values("LCIA method")

,dataset name,dataset reference product,dataset unit,dataset location,LCIA method,impact
91,"electricity production, photovoltaic, commercial","electricity, low voltage",kilowatt hour,AT,"ReCiPe 2016 v1.03, midpoint (H), acidification...",5.328961e-05
92,"electricity production, photovoltaic, commercial","electricity, low voltage",kilowatt hour,AT,"ReCiPe 2016 v1.03, midpoint (H), ecotoxicity: ...",1.834698e-03
93,"electricity production, photovoltaic, commercial","electricity, low voltage",kilowatt hour,AT,"ReCiPe 2016 v1.03, midpoint (H), ecotoxicity: ...",2.523749e-03
94,"electricity production, photovoltaic, commercial","electricity, low voltage",kilowatt hour,AT,"ReCiPe 2016 v1.03, midpoint (H), ecotoxicity: ...",1.825956e-01
95,"electricity production, photovoltaic, commercial","electricity, low voltage",kilowatt hour,AT,"ReCiPe 2016 v1.03, midpoint (H), energy resour...",1.019137e-03
96,"electricity production, photovoltaic, commercial","electricity, low voltage",kilowatt hour,AT,"ReCiPe 2016 v1.03, midpoint (H), eutrophicatio...",2.609507e-06
97,"electricity production, photovoltaic, commercial","electricity, low voltage",kilowatt hour,AT,"ReCiPe 2016 v1.03, midpoint (H), eutrophicatio...",7.461611e-07
98,"electricity production, photovoltaic, commercial","electricity, low voltage",kilowatt hour,AT,"ReCiPe 2016 v1.03, midpoint (H), human toxicit...",5.048081e-03
99,"electricity production, photovoltaic, commercial","electricity, low voltage",kilowatt hour,AT,"ReCiPe 2016 v1.03, midpoint (H), human toxicit...",2.553091e-02
100,"electricity production, photovoltaic, commercial","electricity, low voltage",kilowatt hour,AT,"ReCiPe 2016 v1.03, midpoint (H), ionising radi...",2.539635e-04


In [9]:
I.cost_results

{'SE': {2020: <xarray.DataArray 'cost' (REMIND index: 45, region: 12, impact category: 13)> Size: 56kB
  array([[[3.21839238e-03, 1.48162432e-04, 2.01966533e+00, ...,
           6.03053001e-05, 9.54127950e-04, 3.96262557e-03],
          [3.22009020e-03, 1.48223364e-04, 2.02002716e+00, ...,
           6.03367631e-05, 9.54631695e-04, 3.96390617e-03],
          [3.21834200e-03, 1.48179510e-04, 2.01980484e+00, ...,
           6.03372387e-05, 9.54231065e-04, 3.96388745e-03],
          ...,
          [3.21952828e-03, 1.48333822e-04, 2.01990590e+00, ...,
           6.03338817e-05, 9.55202074e-04, 3.96393872e-03],
          [3.21958445e-03, 1.48325831e-04, 2.02006861e+00, ...,
           6.03397230e-05, 9.55078359e-04, 3.96399105e-03],
          [3.21854304e-03, 1.48167662e-04, 2.01971973e+00, ...,
           6.03407300e-05, 9.54180474e-04, 3.96378875e-03]],
  
         [[2.92976824e-04, 8.75484076e-06, 1.08768295e-01, ...,
           1.19518538e-05, 6.12015801e-05, 2.63640146e-04],
          

In [11]:
mapping[mapping["REMIND index"] == "spv"]

,REMIND index,dataset name,dataset reference product,dataset unit,share,region
51,spv,"electricity production, photovoltaic, commercial","electricity, low voltage",kilowatt hour,1.0,CAZ
116,spv,"electricity production, photovoltaic, commercial","electricity, low voltage",kilowatt hour,1.0,CHA
181,spv,"electricity production, photovoltaic, commercial","electricity, low voltage",kilowatt hour,1.0,EUR
246,spv,"electricity production, photovoltaic, commercial","electricity, low voltage",kilowatt hour,1.0,IND
311,spv,"electricity production, photovoltaic, commercial","electricity, low voltage",kilowatt hour,1.0,JPN
376,spv,"electricity production, photovoltaic, commercial","electricity, low voltage",kilowatt hour,1.0,LAM
441,spv,"electricity production, photovoltaic, commercial","electricity, low voltage",kilowatt hour,1.0,MEA
506,spv,"electricity production, photovoltaic, commercial","electricity, low voltage",kilowatt hour,1.0,NEU
571,spv,"electricity production, photovoltaic, commercial","electricity, low voltage",kilowatt hour,1.0,OAS
636,spv,"electricity production, photovoltaic, commercial","electricity, low voltage",kilowatt hour,1.0,REF


In [7]:
setup = default_setup(I.gdxpath)
mapping = setup["SE"]["mapping"]
regionalized_costs = pd.read_csv("lca/remind/SSP2-NPi-internalize-test-iterative-postsolve/2050/regionalized_costs.csv")

shares = mapping
costs = regionalized_costs

shares = shares.set_index(["dataset name", "dataset reference product", "dataset unit", "region"])
costs = costs.set_index(["dataset name", "dataset reference product", "dataset unit", "region"])

dflist = []
costs_index = costs.index
for idx, row in shares.iterrows():
    # print(idx)
    tech = row["REMIND index"]
    factor = row["share"]
    j = idx
    if idx not in costs_index:
        # print("Using World index")
        j = (idx[0], idx[1], idx[2], "World")
    try:
        sel = costs.loc[j].copy()
    except KeyError:
        print(idx)
        print("Costs not found")
        break
    sel["cost"] = factor * sel["cost"]
    sel = sel.pivot(columns="impact category", values="cost").reset_index(drop=True)
    sel["REMIND index"] = tech
    sel["region"] = idx[-1]
    dflist.append(sel)
    
# return pd.concat(dflist, axis=0, ignore_index=True).groupby(["REMIND index", "region"]).sum()

In [8]:
combine_shares_and_costs(mapping, regionalized_costs)

/p/tmp/davidba/internalizer/internalizer/regionalization.py:294: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sel["cost"] = factor * sel["cost"]
/p/tmp/davidba/internalizer/internalizer/regionalization.py:294: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sel["cost"] = factor * sel["cost"]
/p/tmp/davidba/internalizer/internalizer/regionalization.py:294: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See th

impact category      acidification  climate change  ecotoxicity  \
REMIND index region                                               
MeOH         CAZ          0.000930        0.000048     0.401381   
             CHA          0.000929        0.000048     0.399796   
             EUR          0.000929        0.000048     0.399566   
             IND          0.000929        0.000048     0.400215   
             JPN          0.000930        0.000048     0.400089   
...                            ...             ...          ...   
windoff      NEU          0.000140        0.000003     0.057209   
             OAS          0.000140        0.000003     0.057364   
             REF          0.000140        0.000003     0.057364   
             SSA          0.000140        0.000003     0.057364   
             USA          0.000140        0.000003     0.057364   

impact category      eutrophication  fossil resources  human toxicity  \
REMIND index region                                                     
MeOH         CAZ           0.000004      1.700832e-06        0.000096   
             CHA           0.000004      1.700057e-06        0.000095   
             EUR           0.000004      1.699781e-06        0.000095   
             IND           0.000004      1.700580e-06        0.000095   
             JPN           0.000004      1.700917e-06        0.000095   
...                             ...               ...             ...   
windoff      NEU           0.001959      1.421479e-07        0.000011   
             OAS           0.001960      1.421296e-07        0.000011   
             REF           0.001960      1.421297e-07        0.000011   
             SSA           0.001960      1.421297e-07        0.000011   
             USA           0.001960      1.421297e-07        0.000011   

impact category      ionizing radiation  land use  metal/mineral resources  \
REMIND index region                                                          
MeOH         CAZ           5.828452e-07  0.000066                 0.000046   
             CHA           5.824683e-07  0.000066                 0.000046   
             EUR           5.824118e-07  0.000066                 0.000046   
             IND           5.825468e-07  0.000066                 0.000046   
             JPN           5.827092e-07  0.000066                 0.000046   
...                                 ...       ...                      ...   
windoff      NEU           6.056656e-08  0.000006                 0.000003   
             OAS           6.054176e-08  0.000006                 0.000003   
             REF           6.054177e-08  0.000006                 0.000003   
             SSA           6.054177e-08  0.000006                 0.000003   
             USA           6.054177e-08  0.000006                 0.000003   

impact category      ozone depletion  particulate matter formation  \
REMIND index region                                                  
MeOH         CAZ            2.053778                      0.000062   
             CHA            2.053110                      0.000062   
             EUR            2.053045                      0.000062   
             IND            2.053133                      0.000062   
             JPN            2.052893                      0.000062   
...                              ...                           ...   
windoff      NEU            1.040153                      0.000009   
             OAS            1.043237                      0.000009   
             REF            1.043237                      0.000009   
             SSA            1.043237                      0.000009   
             USA            1.043237                      0.000009   

impact category      photochemical oxidant formation  water use  
REMIND index region                                              
MeOH         CAZ                            0.000325   0.002788  
             CHA                            0.000325  

In [12]:
all_ics = IMPACT_CATEGORIES_MC
ics = []
exclude_list = []
ics = [ic for ic in all_ics if ic not in exclude_list]

# output files
ramp_up_start = 2020
ramp_up_end = 2030
I.write_remind_input_files(
    ramp_up_start,
    ramp_up_end,
    ics
)

In [19]:
sel = regionalized_costs[regionalized_costs["region"] == "EUR"]
sel = sel[sel["dataset name"].str.contains("photovoltaic")]
sel

,dataset name,dataset reference product,dataset unit,region,impact category,cost
3367,"electricity production, photovoltaic, commercial","electricity, low voltage",kilowatt hour,EUR,acidification,2.187526e-04
3368,"electricity production, photovoltaic, commercial","electricity, low voltage",kilowatt hour,EUR,climate change,6.933849e-06
3369,"electricity production, photovoltaic, commercial","electricity, low voltage",kilowatt hour,EUR,ecotoxicity,7.471035e-02
3370,"electricity production, photovoltaic, commercial","electricity, low voltage",kilowatt hour,EUR,eutrophication,2.529583e-03
3371,"electricity production, photovoltaic, commercial","electricity, low voltage",kilowatt hour,EUR,fossil resources,2.533383e-07
3372,"electricity production, photovoltaic, commercial","electricity, low voltage",kilowatt hour,EUR,human toxicity,2.320934e-05
3373,"electricity production, photovoltaic, commercial","electricity, low voltage",kilowatt hour,EUR,ionizing radiation,8.302694e-08
3374,"electricity production, photovoltaic, commercial","electricity, low voltage",kilowatt hour,EUR,land use,1.015266e-05
3375,"electricity production, photovoltaic, commercial","electricity, low voltage",kilowatt hour,EUR,metal/mineral resources,5.959147e-06
3376,"electricity production, photovoltaic, commercial","electricity, low voltage",kilowatt hour,EUR,ozone depletion,3.674396e-01


In [16]:
sel.sum(axis=0)

dataset name                 electricity production, photovoltaic, commerci...
dataset reference product    electricity, low voltageelectricity, low volta...
dataset unit                 kilowatt hourkilowatt hourkilowatt hourkilowat...
region                                 EUREUREUREUREUREUREUREUREUREUREUREUREUR
impact category              acidificationclimate changeecotoxicityeutrophi...
cost                                                                  0.445231
dtype: object

## Costs are off; let's compare the regionalized costs

In [12]:
costs_new = pd.read_csv("/p/tmp/davidba/internalizer/dev/lca/remind/SSP2-NPi-internalize-test-iterative-postsolve/2050/regionalized_costs.csv")
costs_old = pd.read_csv("/p/tmp/davidba/internalization/internalizer/output/SSP2-NPi-internalizeEI-coupled-run0_2025-02-04_17.41.53/remind/SSP2-NPi/2050/regionalized_costs.csv")
costs_old = costs_old[costs_old["quantile"] == 0.5].drop(columns="quantile")

In [13]:
A = costs_old.set_index(["dataset name", "dataset reference product", "dataset unit", "region", "impact category"])
B = costs_new.set_index(["dataset name", "dataset reference product", "dataset unit", "region", "impact category"])

shared_index = A.index.intersection(B.index)

A = A.loc[shared_index]
B = B.loc[shared_index]

reldiff = (B-A) / A

In [16]:
avg = reldiff.reset_index().groupby(["region", "impact category"]).agg({"cost": "median"})
avg.reset_index().sort_values("cost").to_csv("temp.csv")

In [19]:
C = costs_old.groupby(["dataset name", "dataset reference product", "dataset unit", "region"]).agg({"cost": "sum"})
D = costs_new.groupby(["dataset name", "dataset reference product", "dataset unit", "region"]).agg({"cost": "sum"})

shared_index = C.index.intersection(D.index)

C = C.loc[shared_index]
D = D.loc[shared_index]

reldiff = (D-C) / C

avg = reldiff.reset_index().groupby(["dataset name", "region"]).agg({"cost": "median"})
avg.reset_index().sort_values("cost").to_csv("temp2.csv")